In [1]:
# Loading the necessary libraries
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
import requests
import os
import warnings
warnings.filterwarnings("ignore")


In [2]:
# Pulling the qualifying data from Fast F1 API
china_quali = {
    "ANT": (24.003, 27.664, 40.387, 92.064,  1),
    "RUS": (24.012, 27.783, 40.491, 92.286,  2),
    "HAM": (24.080, 27.696, 40.535, 92.415,  3),
    "LEC": (24.022, 27.660, 40.650, 92.428,  4),
    "PIA": (24.120, 27.729, 40.493, 92.550,  5),
    "NOR": (23.995, 27.747, 40.748, 92.608,  6),
    "GAS": (24.099, 27.788, 40.900, 92.873,  7),
    "VER": (24.280, 27.975, 40.613, 93.002,  8),
    "HAD": (24.465, 27.933, 40.659, 93.121,  9),
    "BEA": (24.234, 27.843, 40.931, 93.292, 10),
    "HUL": (24.558, 27.937, 40.743, 93.238, 11),
    "COL": (24.254, 28.078, 40.947, 93.279, 12),
    "OCO": (24.335, 28.041, 41.028, 93.404, 13),
    "LAW": (24.339, 28.117, 40.911, 93.367, 14),
    "LIN": (24.319, 28.181, 40.903, 93.403, 15),
    "BOR": (24.539, 28.145, 40.796, 93.480, 16),
    "SAI": (24.465, 28.669, 41.183, 94.317, 17),
    "ALB": (24.526, 28.694, 41.370, 94.590, 18),
    "ALO": (24.782, 28.723, 41.698, 95.203, 19),
    "BOT": (24.949, 28.972, 41.515, 95.436, 20),
    "STR": (24.953, 29.144, 41.838, 95.935, 21),
    "PER": (25.703, 29.246, 41.611, 96.560, 22),
}

In [3]:
# Adding columns to our dataframe
df = pd.DataFrame.from_dict(
    china_quali, orient="index",
    columns=["S1", "S2", "S3", "ChinaQuali_s", "ChinaGrid"]
)
df.index.name = "Driver"
df = df.reset_index()

In [4]:
df.head()

,Driver,S1,S2,S3,ChinaQuali_s,ChinaGrid
0,ANT,24.003,27.664,40.387,92.064,1
1,RUS,24.012,27.783,40.491,92.286,2
2,HAM,24.080,27.696,40.535,92.415,3
3,LEC,24.022,27.660,40.650,92.428,4
4,PIA,24.120,27.729,40.493,92.550,5


In [5]:
# Merging sector times and adding the duration from the fastes lap
df["UltimateLap_s"] = df["S1"] + df["S2"] + df["S3"]
pole = df["ChinaQuali_s"].min()
df["ChinaGapFromPole_s"] = (df["ChinaQuali_s"] - pole).round(3)

In [6]:
df.head()

,Driver,S1,S2,S3,ChinaQuali_s,ChinaGrid,UltimateLap_s,ChinaGapFromPole_s
0,ANT,24.003,27.664,40.387,92.064,1,92.054,0.000
1,RUS,24.012,27.783,40.491,92.286,2,92.286,0.222
2,HAM,24.080,27.696,40.535,92.415,3,92.311,0.351
3,LEC,24.022,27.660,40.650,92.428,4,92.332,0.364
4,PIA,24.120,27.729,40.493,92.550,5,92.342,0.486


In [7]:
# Comparing the Australian grid positions and finishing positions
aus_grid = {
    "RUS": 1,  "ANT": 2,  "HAD": 3,  "LEC": 4,  "PIA": 5,
    "NOR": 6,  "HAM": 7,  "LAW": 8,  "LIN": 9,  "BOR": 10,
    "OCO": 11, "HUL": 12, "ALB": 13, "GAS": 14, "COL": 15,
    "BEA": 16, "ALO": 17, "PER": 18, "BOT": 19, "VER": 20,
    "SAI": 21, "STR": 22,
}
df["AusGrid"] = df["Driver"].map(aus_grid)

In [8]:
aus_finish_pos = {
    "RUS": 1,  "ANT": 2,  "LEC": 3,  "HAM": 4,
    "NOR": 5,  "VER": 6,  "BEA": 7,  "LIN": 8,
    "BOR": 9,  "GAS": 10, "OCO": 11, "ALB": 12,
    "LAW": 13, "COL": 14, "SAI": 15, "PER": 16,
    "STR": 17,
    "ALO": 20, "BOT": 20, "HAD": 20,
    "PIA": 21, "HUL": 21,
}
df["AusFinishPos"] = df["Driver"].map(aus_finish_pos)

In [9]:
df.head()

,Driver,S1,S2,S3,ChinaQuali_s,ChinaGrid,UltimateLap_s,ChinaGapFromPole_s,AusGrid,AusFinishPos
0,ANT,24.003,27.664,40.387,92.064,1,92.054,0.000,2,2
1,RUS,24.012,27.783,40.491,92.286,2,92.286,0.222,1,1
2,HAM,24.080,27.696,40.535,92.415,3,92.311,0.351,7,4
3,LEC,24.022,27.660,40.650,92.428,4,92.332,0.364,4,3
4,PIA,24.120,27.729,40.493,92.550,5,92.342,0.486,5,21


In [10]:
# Incorporating the Sprint race
SPRINT_RACING_LAPS = 15
sprint_gaps = {
    "RUS": 0.000, "LEC": 0.674, "HAM": 2.554, "NOR": 4.433,
    "ANT": 5.688, "PIA": 6.809, "LAW": 10.900, "BEA": 11.271,
    "VER": 11.619, "OCO": 13.887, "GAS": 14.780, "SAI": 15.753,
    "BOR": 15.858, "COL": 16.393, "HAD": 16.430, "ALB": 20.014,
    "ALO": 21.599, "STR": 21.971, "PER": 28.241,
    "HUL": 35.0, "BOT": 35.0, "LIN": 35.0,
}

In [11]:
# Adjusting the sprint gaps by removing penalties because they inflate the final gap times unfairly
sprint_penalties = {"ANT": 10.0, "PER": 5.0}
sprint_adj = {
    d: max(gap - sprint_penalties.get(d, 0), 0)
    for d, gap in sprint_gaps.items()
}
df["SprintGapPerLap_s"] = df["Driver"].map(
    {d: round(g / SPRINT_RACING_LAPS, 4) for d, g in sprint_adj.items()}
)

In [12]:
df.head()

,Driver,S1,S2,S3,ChinaQuali_s,ChinaGrid,UltimateLap_s,ChinaGapFromPole_s,AusGrid,AusFinishPos,SprintGapPerLap_s
0,ANT,24.003,27.664,40.387,92.064,1,92.054,0.000,2,2,0.0000
1,RUS,24.012,27.783,40.491,92.286,2,92.286,0.222,1,1,0.0000
2,HAM,24.080,27.696,40.535,92.415,3,92.311,0.351,7,4,0.1703
3,LEC,24.022,27.660,40.650,92.428,4,92.332,0.364,4,3,0.0449
4,PIA,24.120,27.729,40.493,92.550,5,92.342,0.486,5,21,0.4539


In [13]:
# Adding and normalizing the team points
team_points = {
    "Mercedes": 55, "Ferrari": 40, "McLaren": 18, "Red Bull": 8,
    "Haas": 7, "Racing Bulls": 6, "Audi": 2, "Alpine": 1,
    "Williams": 1, "Cadillac": 1, "Aston Martin": 1,
}
max_pts = max(team_points.values())
team_score = {t: max(p, 0.5) / max_pts for t, p in team_points.items()}


In [14]:
# Mapping the drivers to their respective teams
driver_to_team = {
    "RUS": "Mercedes", "ANT": "Mercedes", "HAM": "Ferrari",  "LEC": "Ferrari",
    "NOR": "McLaren",  "PIA": "McLaren",  "VER": "Red Bull", "HAD": "Red Bull",
    "BEA": "Haas",     "OCO": "Haas",     "LAW": "Racing Bulls", "LIN": "Racing Bulls",
    "HUL": "Audi",     "BOR": "Audi",     "GAS": "Alpine",   "COL": "Alpine",
    "SAI": "Williams", "ALB": "Williams", "PER": "Cadillac", "BOT": "Cadillac",
    "ALO": "Aston Martin", "STR": "Aston Martin",
}
df["Team"]      = df["Driver"].map(driver_to_team)
df["TeamScore"] = df["Team"].map(team_score)

In [15]:
df.head()

,Driver,S1,S2,S3,ChinaQuali_s,ChinaGrid,UltimateLap_s,ChinaGapFromPole_s,AusGrid,AusFinishPos,SprintGapPerLap_s,Team,TeamScore
0,ANT,24.003,27.664,40.387,92.064,1,92.054,0.000,2,2,0.0000,Mercedes,1.000000
1,RUS,24.012,27.783,40.491,92.286,2,92.286,0.222,1,1,0.0000,Mercedes,1.000000
2,HAM,24.080,27.696,40.535,92.415,3,92.311,0.351,7,4,0.1703,Ferrari,0.727273
3,LEC,24.022,27.660,40.650,92.428,4,92.332,0.364,4,3,0.0449,Ferrari,0.727273
4,PIA,24.120,27.729,40.493,92.550,5,92.342,0.486,5,21,0.4539,McLaren,0.327273


In [16]:
# Rainfall and temperature conditions

rain_probability = 0.07
temperature = 13.0
df["RainProbability"] = rain_probability
df["Temperature"] = temperature

In [17]:
df.head(10)

,Driver,S1,S2,S3,ChinaQuali_s,ChinaGrid,UltimateLap_s,ChinaGapFromPole_s,AusGrid,AusFinishPos,SprintGapPerLap_s,Team,TeamScore,RainProbability,Temperature
0,ANT,24.003,27.664,40.387,92.064,1,92.054,0.000,2,2,0.0000,Mercedes,1.000000,0.07,13.0
1,RUS,24.012,27.783,40.491,92.286,2,92.286,0.222,1,1,0.0000,Mercedes,1.000000,0.07,13.0
2,HAM,24.080,27.696,40.535,92.415,3,92.311,0.351,7,4,0.1703,Ferrari,0.727273,0.07,13.0
3,LEC,24.022,27.660,40.650,92.428,4,92.332,0.364,4,3,0.0449,Ferrari,0.727273,0.07,13.0
4,PIA,24.120,27.729,40.493,92.550,5,92.342,0.486,5,21,0.4539,McLaren,0.327273,0.07,13.0
5,NOR,23.995,27.747,40.748,92.608,6,92.490,0.544,6,5,0.2955,McLaren,0.327273,0.07,13.0
6,GAS,24.099,27.788,40.900,92.873,7,92.787,0.809,14,10,0.9853,Alpine,0.018182,0.07,13.0
7,VER,24.280,27.975,40.613,93.002,8,92.868,0.938,20,6,0.7746,Red Bull,0.145455,0.07,13.0
8,HAD,24.465,27.933,40.659,93.121,9,93.057,1.057,3,20,1.0953,Red Bull,0.145455,0.07,13.0
9,BEA,24.234,27.843,40.931,93.292,10,93.008,1.228,16,7,0.7514,Haas,0.127273,0.07,13.0


In [18]:
#Normalizing Australian positions and sprint gap
aus_pos_norm    = (df["AusFinishPos"]     - 1) / 21
sprint_gap_norm = df["SprintGapPerLap_s"] / df["SprintGapPerLap_s"].max()

df["RaceScore"] = (0.5 * aus_pos_norm + 0.5 * sprint_gap_norm).round(4)

In [19]:
df.head(10)

,Driver,S1,S2,S3,ChinaQuali_s,ChinaGrid,UltimateLap_s,ChinaGapFromPole_s,AusGrid,AusFinishPos,SprintGapPerLap_s,Team,TeamScore,RainProbability,Temperature,RaceScore
0,ANT,24.003,27.664,40.387,92.064,1,92.054,0.000,2,2,0.0000,Mercedes,1.000000,0.07,13.0,0.0238
1,RUS,24.012,27.783,40.491,92.286,2,92.286,0.222,1,1,0.0000,Mercedes,1.000000,0.07,13.0,0.0000
2,HAM,24.080,27.696,40.535,92.415,3,92.311,0.351,7,4,0.1703,Ferrari,0.727273,0.07,13.0,0.1079
3,LEC,24.022,27.660,40.650,92.428,4,92.332,0.364,4,3,0.0449,Ferrari,0.727273,0.07,13.0,0.0572
4,PIA,24.120,27.729,40.493,92.550,5,92.342,0.486,5,21,0.4539,McLaren,0.327273,0.07,13.0,0.5735
5,NOR,23.995,27.747,40.748,92.608,6,92.490,0.544,6,5,0.2955,McLaren,0.327273,0.07,13.0,0.1586
6,GAS,24.099,27.788,40.900,92.873,7,92.787,0.809,14,10,0.9853,Alpine,0.018182,0.07,13.0,0.4254
7,VER,24.280,27.975,40.613,93.002,8,92.868,0.938,20,6,0.7746,Red Bull,0.145455,0.07,13.0,0.2850
8,HAD,24.465,27.933,40.659,93.121,9,93.057,1.057,3,20,1.0953,Red Bull,0.145455,0.07,13.0,0.6871
9,BEA,24.234,27.843,40.931,93.292,10,93.008,1.228,16,7,0.7514,Haas,0.127273,0.07,13.0,0.3039


In [20]:
# Feature engineering
feature_cols = [
    "UltimateLap_s",       # China quali: best S1+S2+S3 — pure car pace
    "ChinaGapFromPole_s",  # China quali: gap to pole — single lap delta
    "ChinaGrid",           # China qualifying position
    "AusGrid",             # Australia qualifying position — cross-circuit pace
    "TeamScore",           # 2026 constructor standings
    "RainProbability",     # weather
    "Temperature",         # weather
]

X = df[feature_cols].copy()
y = df["RaceScore"]

In [21]:
# Filling any missing values with median
imputer   = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)

In [22]:
# The Gradient Boost model
model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.5,
    random_state=42,
)

loo        = LeaveOneOut()
loo_errors = []
for train_idx, test_idx in loo.split(X_imputed):
    X_tr, X_te = X_imputed[train_idx], X_imputed[test_idx]
    y_tr, y_te = y.iloc[train_idx],    y.iloc[test_idx]
    model.fit(X_tr, y_tr)
    loo_errors.append(abs(model.predict(X_te)[0] - y_te.iloc[0]))

loo_mae = np.mean(loo_errors)

model.fit(X_imputed, y)
df["PredictedScore"] = model.predict(X_imputed)

final = df.sort_values("PredictedScore").reset_index(drop=True)

In [23]:
# Printing the predicted reults
print("\n" + "=" * 68)
print("   PREDICTED FINISHING ORDER — 2026 CHINESE GRAND PRIX")
print("   Target: Australia finish pos + China sprint gap/lap")
print("=" * 68)
print(f"  {'P':<4}{'DRV':<6}{'Team'}")
print("  " + "─" * 20)
for i, row in final.iterrows():
    print(f"  {i+1:<4}{row['Driver']:<6}{row['Team']}")

podium = final.head(3)
print(f"\n  {'='*45}")
print(f"  🥇 P1: {podium.iloc[0]['Driver']} ({podium.iloc[0]['Team']})")
print(f"  🥈 P2: {podium.iloc[1]['Driver']} ({podium.iloc[1]['Team']})")
print(f"  🥉 P3: {podium.iloc[2]['Driver']} ({podium.iloc[2]['Team']})")
print(f"\n  Leave One out MAE: {loo_mae:.4f}")
print(f"  Weather: {temperature:.1f}°C | Rain: {rain_probability:.0%}")
print(f"  {'='*45}")

print("\nFeature importances:")
for feat, imp in sorted(zip(feature_cols, model.feature_importances_),
                         key=lambda x: -x[1]):
    bar = "█" * int(imp * 50)
    print(f"  {feat:<22} {imp:.4f}  {bar}")


   PREDICTED FINISHING ORDER — 2026 CHINESE GRAND PRIX
   Target: Australia finish pos + China sprint gap/lap
  P   DRV   Team
  ────────────────────
  1   RUS   Mercedes
  2   ANT   Mercedes
  3   LEC   Ferrari
  4   HAM   Ferrari
  5   NOR   McLaren
  6   VER   Red Bull
  7   BEA   Haas
  8   GAS   Alpine
  9   OCO   Haas
  10  BOR   Audi
  11  LAW   Racing Bulls
  12  PIA   McLaren
  13  SAI   Williams
  14  COL   Alpine
  15  ALB   Williams
  16  LIN   Racing Bulls
  17  HAD   Red Bull
  18  PER   Cadillac
  19  STR   Aston Martin
  20  ALO   Aston Martin
  21  BOT   Cadillac
  22  HUL   Audi

  🥇 P1: RUS (Mercedes)
  🥈 P2: ANT (Mercedes)
  🥉 P3: LEC (Ferrari)

  Leave One out MAE: 0.1958
  Weather: 13.0°C | Rain: 7%

Feature importances:
  TeamScore              0.3541  █████████████████
  ChinaGrid              0.2281  ███████████
  ChinaGapFromPole_s     0.1989  █████████
  UltimateLap_s          0.1707  ████████
  AusGrid                0.0481  ██
  RainProbability        0.00